In [ ]:
import faiss
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma



# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


# Chat model
model = ChatOllama(model="qwen3")

"""
            faiss index for vector search using HNSW (Hierarchical Navigable Small World) 
            appaximate search algorithm
"""
dimension = 384  # Dimension of the embeddings
M = 16  # Number of neighbors to consider in HNSW
faiss_index = faiss.IndexHNSWFlat(
    dimension,
    M,
    faiss.METRIC_INNER_PRODUCT
)

# Higher value = better recall, but slower indexing
faiss_index.hnsw.efConstruction = 200

# Higher value = better recall, but slower search
faiss_index.hnsw.efSearch = 64


# Chunking the document into smaller pieces for better retrieval
file_path = r"D:\RAG\VectotDatabases\FASIS\RAG_pipeline_2\doc\llama2-research-paper.pdf"

loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print("Total pages:", len(pages))
print("Total chunks:", len(chunks))

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=False
)

vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

#print(retriever.invoke("What is the architecture of Llama 1?"))
#exit(1)



def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the supplied context.
    If the answer is not available in the context, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

###################################### RAG Chain ############################
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

# --------------------------------------------------
# 12. Ask a question related to the PDF
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the algorithm of tokenizer used ?"
)
print(answer)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3001.99it/s]


Total pages: 77
Total chunks: 343
The tokenizer used employs a **byte pair encoding (BPE)** algorithm, as specified in the context. This implementation is based on the work by Sennrich et al. (2016) and utilizes the SentencePiece library (Kudo and Richardson, 2018). 

**Answer:**  
The tokenizer uses the **byte pair encoding (BPE)** algorithm.
